In [ ]:
#| default_exp handlers.titanica_fram_strait

# Titanica - Radionuclide Handler

Generic handler for Titanica community records on Zenodo. Converts wide-format
seawater radionuclide tables (CSV, Excel, or ZIP) to MARIS-standard NetCDF4.

**Auto-detection strategy** (no per-record YAML config required):

| What | How |
|------|-----|
| File format | URL path extension (`.csv` / `.zip` / `.xlsx`) |
| Nuclide columns | Regex `VAL_RE` built from `NUCLIDE_LUT` keys at import time |
| Unit (at/L or at/kg) | `UNITS_LUT` token in column name; MARIS id resolved automatically |
| Scale factor (x10^N) | Suffix `( x 10^N)` parsed from column name |
| Date column | `Date` column, or `Date_Time` as fallback |
| Datetime format | Anchor regex on first non-null date value; 4 patterns supported |
| Time column | First of `['Time', 'Time (UTC)']` present; omitted for combined/date-only |

Validated on all 7 measurement records in the Titanica Zenodo community:
- **19387002** Fram Strait 2020/2021 - CSV, European DD/MM/YYYY date
- **18880777** JOIS 2022 - ZIP + Excel, scale x10^7/x10^6, ISO date + time col
- **18880591** JOIS 2023 - ZIP + Excel, pandas datetime Date col, NaT rows handled
- **18880497** JOIS 2024 - ZIP + Excel, same structure
- **18880401** JOIS 2021 - ZIP + Excel, space-separated x10^N suffix
- **16914587** Davis Strait - XLSX, at_kg unit, quoted month-name date, no time col
- **15056897** ODEN SO21 - CSV, Date_Time combined col, Latitude/Longitude names

In [ ]:
#| export
from fastcore.all import *
import pandas as pd
import numpy as np
import re
import requests
import zipfile
import io

from marisco.callbacks import (PerGroupCB, Callback, Transformer, EncodeTimeCB,
                                SanitizeLonLatCB, RemapCB, AddSampleIDCB)
from marisco.metadata import GlobAttrsFeeder, BboxCB, DepthRangeCB, TimeRangeCB, InisCB, KeyValuePairCB
from marisco.configs import get_lut
from marisco.encoders import NetCDFEncoder

## Constants & LUT values

In [ ]:
#| export
# InvenioRDM base URL (mirrors the INIS API pattern used by InisCB)
ZENODO_API = 'https://zenodo-rdm.web.cern.ch/api/records'

# Detection limit status: 1 = measured (detected value)
DL_DETECTED = 1

# MARIS area IDs resolved once at import time from dbo_area.xlsx
_area_lut          = get_lut('AREA')
AREA_GREENLAND_SEA = _area_lut['Greenland Sea']  # 2356
AREA_BEAUFORT_SEA  = _area_lut['Beaufort Sea']   # 4256

# Complete MARIS seawater-schema column whitelist
MARIS_SEA_COLS = (
    'LAT', 'LON', 'TIME', 'SMP_DEPTH', 'TOT_DEPTH',
    'SMP_ID_PROVIDER',
    'TEMP', 'SAL', 'OXYGEN', 'PH', 'TURBIDITY',
    'NUCLIDE', 'VALUE', 'UNC', 'UNIT', 'LAB', 'DL', 'AREA',
)

## Fram Strait reference-record fixtures

Notebook-only values for interactive exploration.
**Not exported** — the handler is generic and must work for any Titanica record.
Always pass these (or equivalent) explicitly when calling `encode()` or `load_data()`.

In [ ]:
# Fram Strait 2020-2021 reference dataset -- notebook exploration only, not exported.
CSV_URL = (
    'https://zenodo-rdm.web.cern.ch/api/records/19387002'
    '/files/FramStrait_2020_2021_radionuclides.csv/content'
)
fname_out  = '../../_data/output/titanica-fram-strait.nc'
zenodo_key = '19387002'
kw = [
    'I-129', 'U-236', 'radionuclides', 'seawater',
    'Fram Strait', 'Arctic Ocean', 'marine radioactivity',
    'FS20', 'FS21', 'Norwegian Polar Institute',
]

## Regex pattern & LUT dictionaries

`MeltByPatternCB` uses `VAL_RE` to scan column names and auto-resolves
nuclide IDs, lab IDs, unit IDs, and scale factors — with no per-record
configuration.

`VAL_RE` is assembled **dynamically** from `NUCLIDE_LUT`, `UNITS_LUT`, and
`LAB_LUT` at module load time. Adding a new nuclide requires only one dict
entry in `NUCLIDE_LUT`; the regex updates automatically with zero further edits.

### Column name anatomy

```
ETH_U236_at_l ( x 10^6)
│   │    │       │
│   │    unit    scale suffix (optional)
│   nuclide token (matched case-insensitively via NUCLIDE_LUT keys)
lab prefix (optional, matched via LAB_LUT keys)
```

`unc_` columns are identified by replacing `<lab_>?<nuc>` with `<lab_>?unc_<nuc>`:
`I129_at_l ( x 10^7)` → `unc_I129_at_l ( x 10^7)`

In [ ]:
#| export
# Nuclide nc_name -> MARIS nuclide_id (dbo_nuclide.xlsx).
# Defined first so VAL_RE can be compiled from its keys.
# To support a new nuclide add one entry here; VAL_RE updates automatically.
_nuc_lut = get_lut('NUCLIDE')
NUCLIDE_LUT = {
    'i129': _nuc_lut['i129'],  # 28
    'u236': _nuc_lut['u236'],  # 108
}

# Unit token -> {MARIS unit_id, base conversion factor}.
# Scale factors embedded in column names like '( x 10^7)' are extracted separately
# and multiplied on top of 'factor', so datasets sharing a unit need only one entry.
UNITS_LUT = {
    'at_l':  {'id': 12, 'factor': 1.0},  # atoms/L  (dbo_unit unit_id=12)
    'at_kg': {'id':  9, 'factor': 1.0},  # atoms/kg (dbo_unit unit_id=9)
}

# Lab token -> MARIS lab_id (dbo_lab_cleaned.xlsx).
# Empty string '' covers all columns with no lab prefix.
_lab_lut = get_lut('LAB')
LAB_LUT = {
    '':    _lab_lut['Not available'],                                                    # 0
    'ETH': _lab_lut['Laboratory of Ion Beam Physics _LIP_, ETZ Zürich, Switzerland'],   # 345
}

# VAL_RE assembled from the three LUTs above -- zero hardcoded nuclide/unit/lab strings.
# New entries in NUCLIDE_LUT, UNITS_LUT, or LAB_LUT are picked up automatically.
_nuc_pat  = '|'.join(re.escape(k) for k in NUCLIDE_LUT)   # e.g. i129|u236
_unit_pat = '|'.join(re.escape(k) for k in UNITS_LUT)      # e.g. at_l|at_kg
_lab_pat  = '|'.join(re.escape(k) for k in LAB_LUT if k)   # e.g. ETH
VAL_RE = (
    rf'^(?:(?P<lab>{_lab_pat})_)?'
    rf'(?P<nuc>{_nuc_pat})_'
    rf'(?P<unit>{_unit_pat})\b'
)

# Provider column name -> MARIS column name (unified across all Titanica datasets).
# Having two sources map to the same MARIS name is safe: only one will be present per dataset.
COL_RENAME = {
    'Latitude_degN':        'LAT',
    'Longitude_degE':       'LON',
    'Latitude':             'LAT',    # ODEN (15056897)
    'Longitude':            'LON',    # ODEN (15056897)
    'Depth_m':              'SMP_DEPTH',
    'Bottom_Depth_m':       'TOT_DEPTH',
    'DepthWater_m_':        'SMP_DEPTH',   # ODEN (15056897)
    'BottomDepth_m':        'TOT_DEPTH',   # ODEN (15056897)
    'Sample_ID':            'SMP_ID_PROVIDER',
    'sample_number':        'SMP_ID_PROVIDER',   # JOIS 2022
    'Temperature_degC_ctd': 'TEMP',
    'Temperature_degC':     'TEMP',              # JOIS 2022
    'Salinity_ctd':         'SAL',
    'Salinity_psu':         'SAL',               # JOIS 2022
}

## Callbacks

In [ ]:
#| export
# Ordered (anchor_regex, strptime_format, needs_separate_time_col) triples tried by
# ParseDateTimeCB._detect.  Top-to-bottom: first anchor match wins.
# Raising ValueError on no-match prevents silent month/day transposition.
# Two month-name entries are required because Python's strptime treats %b (abbreviated)
# and %B (full) as mutually exclusive -- 'Dec' only parses with %b, 'December' only with %B.
_DATE_FORMATS = [
    (r'^\d{4}-\d{2}-\d{2}T\d{2}', '%Y-%m-%dT%H:%M:%S', False),  # ISO 8601 combined (Date_Time)
    (r'^\d{4}-\d{2}-\d{2}',          '%Y-%m-%d %H:%M:%S', True),   # ISO date + separate time col
    (r'^\d{2}/\d{2}/\d{4}',          '%d/%m/%Y %H:%M:%S', True),   # European DD/MM/YYYY + time col
    (r'^[A-Za-z]{3}\s',              '%b %d %Y',           False),  # 3-char month abbrev (Jan, Dec)
    (r'^[A-Za-z]',                    '%B %d %Y',           False),  # full month name (January)
]

class ParseDateTimeCB(PerGroupCB):
    "Parse provider date + time columns into a single UTC datetime in TIME."
    def __init__(self,
                 col_date: str='Date',   # Provider date column (or combined datetime column)
                 col_time: str=None,     # Provider time column; auto-detected if None
                 col_out:  str='TIME',   # Output datetime column
                 fmt:      str=None,     # strptime format; auto-detected if None
                 ):
        store_attr()

    def _detect(self, df):
        "Return (col_date, col_time_or_none, fmt). Tries Date_Time as fallback when col_date absent. col_time_or_none is None when date column already contains time or when no time column exists. Raises ValueError on ambiguity."
        col_date = self.col_date if self.col_date in df.columns else next(
            (c for c in ('Date_Time',) if c in df.columns), None)
        if col_date is None:
            raise ValueError(
                f"No date column found. Expected '{self.col_date}' or 'Date_Time'; "
                "pass col_date= explicitly to ParseDateTimeCB."
            )
        col_time = self.col_time or next(
            (c for c in ('Time', 'Time (UTC)') if c in df.columns), None)
        if self.fmt is not None:
            needs_time = col_time is not None and 'H' in self.fmt and 'T' not in self.fmt
            return col_date, (col_time if needs_time else None), self.fmt
        raw = str(df[col_date].dropna().iloc[0]).strip()
        sample = raw.strip('"').strip("'")
        for anchor, fmt, needs_time in _DATE_FORMATS:
            if re.match(anchor, sample):
                # When the matched format requires a time col but none exists, trim to date-only.
                # Avoids NaT cascade when pandas reads Date as datetime64 with no companion Time col.
                if needs_time and not col_time: fmt = fmt.partition(' ')[0]
                return col_date, (col_time if (needs_time and col_time) else None), fmt
        raise ValueError(
            f"Cannot auto-detect date format from sample value '{raw}'. "
            f"Known anchors: {[p for p, _, _ in _DATE_FORMATS]}. "
            f"Pass fmt= explicitly to ParseDateTimeCB."
        )

    def each_grp(self, grp, df, tfm):
        col_date, col_time, fmt = self._detect(df)
        # Fast path: pandas already parsed the col as datetime64 and no separate time col to join.
        if pd.api.types.is_datetime64_any_dtype(df[col_date]) and col_time is None:
            tfm.dfs[grp][self.col_out] = pd.to_datetime(df[col_date], utc=True, errors='coerce')
            return
        if pd.api.types.is_datetime64_any_dtype(df[col_date]):
            date_strs = df[col_date].dt.strftime('%Y-%m-%d').fillna('')
        else:
            date_strs = df[col_date].astype(str).str.strip().str.strip('"').str.strip("'")
        if col_time is not None:
            time_strs = df[col_time].astype(str).str.strip()
            combined = date_strs + ' ' + time_strs
        else:
            combined = date_strs
        tfm.dfs[grp][self.col_out] = pd.to_datetime(combined, format=fmt, errors='coerce', utc=True)

In [ ]:
#| export
class RenameAndSelectCB(PerGroupCB):
    """Rename provider columns to MARIS names and drop all non-MARIS columns."""
    def __init__(self,
                 rename: dict=COL_RENAME,
                 keep_maris: tuple=MARIS_SEA_COLS,
                 ):
        store_attr()

    def each_grp(self, grp, df, tfm):
        df = df.rename(columns=self.rename)
        present = [c for c in self.keep_maris if c in df.columns]
        tfm.dfs[grp] = df[present]

In [ ]:
#| export
class MeltByPatternCB(Callback):
    "Detect nuclide value columns by regex, pair with uncertainty counterparts, and reshape wide provider data to MARIS long format."
    def __init__(self,
                 val_re: str=VAL_RE,                 # Regex with named groups: lab?, nuc, unit
                 units_lut: dict=UNITS_LUT,           # Unit token -> {id, factor}
                 lab_lut: dict=LAB_LUT,               # Lab token -> MARIS lab_id
                 nuclide_lut: dict=NUCLIDE_LUT,       # nc_name -> MARIS nuclide_id
                 grp: str='SEAWATER',
                 ): store_attr()

    def _parse_col(self, col):
        """Return per-column metadata dict, or None if col does not match val_re.
        Scale exponent is extracted solely from the column name suffix '( x 10^N)'
        via regex; no record ID, URL, or filename is consulted anywhere."""
        m = re.match(self.val_re, col, re.IGNORECASE)
        if not m: return None
        lab_tok  = (m.group('lab') or '').upper()
        nuc_tok  = m.group('nuc').lower()
        unit_tok = m.group('unit').lower()
        sm        = re.search(r'\(\s*x\s*10\^(\d+)\s*\)', col)
        col_scale = 10 ** int(sm.group(1)) if sm else 1.0
        unit_info = self.units_lut.get(unit_tok, {'id': 0, 'factor': 1.0})
        return {
            'nuc_id':  self.nuclide_lut.get(nuc_tok, 0),
            'lab_id':  self.lab_lut.get(lab_tok, 0),
            'unit_id': unit_info['id'],
            'factor':  col_scale * unit_info['factor'],
            'lab_tok': lab_tok,
        }

    def _unc_col(self, val_col, lab_tok):
        "Derive the uncertainty column name by inserting 'unc_' after any lab prefix."
        prefix = (lab_tok + '_') if lab_tok else ''
        rest   = val_col[len(prefix):]
        return prefix + 'unc_' + rest

    def __call__(self, tfm):
        if self.grp not in tfm.dfs: return
        df, frames = tfm.dfs[self.grp], []
        skip_re = re.compile(r'^(eth_)?unc_', re.IGNORECASE)
        for col in df.columns:
            if skip_re.match(col): continue
            info = self._parse_col(col)
            if not info: continue
            unc_col = self._unc_col(col, info['lab_tok'])
            sub = df.dropna(subset=[col]).copy()
            sub['VALUE']   = sub[col] * info['factor']
            sub['UNC']     = sub[unc_col] * info['factor'] if unc_col in df.columns else np.nan
            sub['NUCLIDE'] = info['nuc_id']
            sub['UNIT']    = info['unit_id']
            sub['LAB']     = info['lab_id']
            frames.append(sub)
        if frames: tfm.dfs[self.grp] = pd.concat(frames, ignore_index=True)

## `get_attrs`, `load_data` & `encode`

In [ ]:
#| export
def get_attrs(
    tfm: Transformer,            # Transformer after pipeline has run
    zenodo_key: str,             # Zenodo record ID
    kw: list=None,               # Keywords (empty list used if None)
    base_url: str=ZENODO_API,    # InvenioRDM / Zenodo API base URL
) -> dict:
    "Retrieve all NetCDF global attributes from Zenodo metadata and spatial/temporal coverage."
    return GlobAttrsFeeder(tfm.dfs, cbs=[
        BboxCB(),
        DepthRangeCB(),
        TimeRangeCB(),
        InisCB(zenodo_key, base_url=base_url),
        KeyValuePairCB('keywords', ', '.join(kw or [])),
        KeyValuePairCB('publisher_postprocess_logs', ', '.join(tfm.logs)),
    ])()

In [ ]:
#| export
def load_data(url: str) -> pd.DataFrame:
    "Fetch Titanica radionuclide data from Zenodo; auto-detects CSV, ZIP+Excel, and XLSX formats from the URL path."
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    raw = r.content
    # Detect file format from the URL path (before any '?' args or '/content' suffix)
    url_path = url.split('?')[0].rstrip('/')
    ext = next((e for e in ('.zip', '.xlsx', '.xls', '.csv') if e in url_path.lower()), '.csv')
    if ext == '.zip':
        with zipfile.ZipFile(io.BytesIO(raw)) as z:
            names = z.namelist()
            csv_names = [n for n in names if n.lower().endswith('.csv')]
            xl_names  = [n for n in names if n.lower().endswith(('.xlsx', '.xls'))]
            if csv_names:
                df = pd.read_csv(io.BytesIO(z.read(csv_names[0])))
            else:
                xl = pd.ExcelFile(io.BytesIO(z.read(xl_names[0])))
                df = xl.parse(xl.sheet_names[0])
    elif ext in ('.xlsx', '.xls'):
        xl = pd.ExcelFile(io.BytesIO(raw))
        df = xl.parse(xl.sheet_names[0])
    else:
        df = pd.read_csv(io.BytesIO(raw))
    print(f'Shape  : {df.shape}')
    print(f'Columns: {df.columns.tolist()}')
    return df

In [ ]:
#| export
def encode(
    fname_out: str,              # Output path for the NetCDF4 file
    url: str,                    # Zenodo file download URL (CSV, ZIP, or Excel)
    zenodo_key: str,             # Zenodo record ID
    kw: list=None,               # Keywords for NetCDF global attributes
    area_id: int=AREA_GREENLAND_SEA,  # MARIS area_id
    **kwargs,
) -> None:
    "Encode a Titanica radionuclide dataset to MARIS-standard NetCDF4."
    dfs = {'SEAWATER': load_data(url)}
    tfm = Transformer(dfs, cbs=[
        ParseDateTimeCB(),
        MeltByPatternCB(),
        RemapCB(lut={}, col_remap='DL',   col_src='NUCLIDE', default_val=DL_DETECTED),
        RemapCB(lut={}, col_remap='AREA', col_src='NUCLIDE', default_val=area_id),
        RenameAndSelectCB(rename=COL_RENAME),
        SanitizeLonLatCB(),
        EncodeTimeCB(),
        AddSampleIDCB(col_provider='SMP_ID_PROVIDER'),
    ])
    tfm()
    encoder = NetCDFEncoder(
        tfm.dfs,
        dest_fname=fname_out,
        global_attrs=get_attrs(tfm, zenodo_key=zenodo_key, kw=kw),
        verbose=kwargs.get('verbose', False),
    )
    encoder.encode()

## NetCDF → CSV (MARIS DB import)

In [ ]:
#|eval: false
# from marisco.netcdf2csv import decode
# decode(fname_in=fname_out, verbose=True)